# Daily Practice — 2026-11-14 — ML Testing & MLOps: Building a Model Promotion Gate

**Dataset:** [Breast Cancer Wisconsin (Diagnostic)](https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic) —
569 real patient biopsies, 30 real-valued features computed from digitized images of fine-needle
aspirate cell nuclei (radius, texture, perimeter, area, smoothness, concavity, ...), labeled
malignant/benign. Loaded via `sklearn.datasets.load_breast_cancer`.

**Problem statement.** You're the QA engineer embedded with an ML team that ships a malignancy
classifier built on this dataset. The team retrains and redeploys often, and a bad retrain has
already reached production three times in the past, each a different flavor of failure:

1. An upstream feature pipeline bug silently changed the *units* of one input column, and nobody
   noticed until predictions looked strange in production.
2. A "better" candidate model shipped because overall accuracy went up — but it quietly traded
   away recall on malignant cases, which is the error that actually costs lives.
3. A model was re-serialized and reloaded from disk, and its predictions on the exact same inputs
   silently changed, because a preprocessing step inside the pipeline wasn't fully deterministic.

Your job is to make sure none of these three failure modes can reach production silently again, by
building an automated **promotion gate**: three independent checks that a new model candidate must
pass before it can replace the model currently in production.

**What you should produce.** Four functions, each returning a small structured report (a `dict`),
plus two deliberately-broken fixtures that prove the gate actually catches failures rather than
rubber-stamping everything:

1. `validate_schema(df, schema=SCHEMA) -> dict` — checks that an inference-time batch has exactly
   the expected feature columns, as numeric dtypes, with no nulls, and with every value inside the
   range recorded in the frozen `SCHEMA` contract (built below from the training data). Returns
   `{"passed": bool, "violations": [list of human-readable strings]}`.
2. `check_regression(candidate_model, baseline_metrics, X_test, y_test, min_recall_drop=-0.02) -> dict` —
   scores the candidate on the malignant class and fails if its **recall** drops by more than
   `min_recall_drop` versus the frozen baseline, even if accuracy went up. Returns
   `{"passed": bool, "candidate_metrics": {...}, "baseline_metrics": {...}, "reason": str}`.
3. `check_determinism(model_path, X_sample) -> dict` — reloads the serialized model from disk with
   `joblib` and confirms its predictions on a fixed sample are identical to the in-memory model's
   predictions. Returns `{"passed": bool, "mismatches": int}`.
4. `run_gate(candidate_model, model_path, X_test, y_test, baseline_metrics, inference_batch) -> dict` —
   runs all three checks, prints a CI-style pass/fail summary, and returns the combined report.
5. A deliberately bad candidate model (e.g. accidentally trained on a class-skewed subsample) that
   **should fail** `check_regression`, and a deliberately corrupted inference batch (e.g. a
   unit-conversion bug) that **should fail** `validate_schema`. Run the full gate on both the good
   and the bad combination and confirm the gate's verdict matches what you'd expect by hand.


## Setup — data, the production baseline, and the frozen contract

Run this section as-is. It plays the role of "what's already in production": a trained baseline
model, its recorded test-set metrics, and a schema contract describing the feature ranges it was
trained on. Your job starts in the next section.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import joblib

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import recall_score, precision_score, accuracy_score

RANDOM_STATE = 42

data = load_breast_cancer(as_frame=True)
df = data.frame.copy().rename(columns={"target": "label"})
# sklearn encodes 0 = malignant, 1 = benign; flip so 1 = malignant (the class we most need to catch)
df["label"] = 1 - df["label"]
feature_cols = [c for c in df.columns if c != "label"]

print(df.shape)
df["label"].value_counts()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df[feature_cols], df["label"], test_size=0.25, stratify=df["label"], random_state=RANDOM_STATE
)

baseline_model = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])
baseline_model.fit(X_train, y_train)

baseline_preds = baseline_model.predict(X_test)
BASELINE_METRICS = {
    "recall": recall_score(y_test, baseline_preds),
    "precision": precision_score(y_test, baseline_preds),
    "accuracy": accuracy_score(y_test, baseline_preds),
}
BASELINE_METRICS

In [ ]:
# The schema "contract": the range each feature is expected to fall in at inference time.
# Built from ALL known-good reference data (not just the train split) with a small margin,
# since a contract built only from one split will false-positive on legitimate new data
# that happens to land a little beyond what that split happened to contain.
SCHEMA_MARGIN = 0.03
SCHEMA = {}
for col in feature_cols:
    lo, hi = float(df[col].min()), float(df[col].max())
    pad = (hi - lo) * SCHEMA_MARGIN
    SCHEMA[col] = {"dtype": "float64", "min": lo - pad, "max": hi + pad}

MODEL_PATH = Path("baseline_model.joblib")
joblib.dump(baseline_model, MODEL_PATH)
print(f"saved production model to {MODEL_PATH.resolve()}")

## Your task — build the gate

Fill in the four functions below. Each should match the return shape described in the problem
statement above. Don't peek at the solution until you've got your own version passing the sanity
checks at the bottom of this section.

In [ ]:
def validate_schema(df_in, schema=SCHEMA):
    # TODO: Check that df_in has exactly `schema`'s columns, as numeric dtypes, with no
    # nulls, and every value inside [schema[col]["min"], schema[col]["max"]].
    # Return {"passed": bool, "violations": [human-readable strings]}.
    # TODO: check for missing / unexpected columns
    # TODO: check dtype of each expected column
    # TODO: check for nulls
    # TODO: check each column's values are within schema[col]["min"]/["max"]
    raise NotImplementedError

In [ ]:
def check_regression(candidate_model, baseline_metrics, X_test, y_test, min_recall_drop=-0.02):
    # TODO: Score candidate_model on X_test/y_test (recall, precision, accuracy on the
    # malignant=1 class). Fail if recall dropped by more than min_recall_drop vs. baseline_metrics,
    # regardless of what happened to accuracy.
    # Return {"passed": bool, "candidate_metrics": {...}, "baseline_metrics": {...}, "reason": str}.
    # TODO: get candidate predictions and compute its metrics dict
    # TODO: compare candidate recall vs. baseline recall against min_recall_drop
    # TODO: build a human-readable `reason` string explaining the comparison
    raise NotImplementedError

In [ ]:
def check_determinism(model_path, X_sample):
    # TODO: Reload the model at model_path with joblib and compare its predictions on
    # X_sample against `baseline_model`'s in-memory predictions on the same rows.
    # Return {"passed": bool, "mismatches": int}.
    # TODO: joblib.load the model from model_path
    # TODO: predict on X_sample with both the reloaded model and the in-memory baseline_model
    # TODO: count mismatches between the two prediction arrays
    raise NotImplementedError

In [ ]:
def run_gate(candidate_model, model_path, X_test, y_test, baseline_metrics, inference_batch, schema=SCHEMA):
    # TODO: Run validate_schema, check_regression, and check_determinism, print a short
    # CI-style PASS/FAIL summary for each plus an overall verdict, and return a dict with
    # all four results: {"overall_passed": bool, "schema": ..., "regression": ..., "determinism": ...}.
    # TODO: call the three checks
    # TODO: combine into overall_passed
    # TODO: print a readable summary
    raise NotImplementedError

### Now build two fixtures that should make the gate fail

- `bad_recall_model`: retrain the same kind of pipeline, but on a training set where you've
  accidentally kept only a small fraction of the malignant examples (simulating an upstream
  sampling bug). It should score similar or better on accuracy but noticeably worse on recall.
- `corrupted_batch`: a copy of `X_test` with one column scaled by some large factor (simulating a
  unit-conversion bug upstream, e.g. mm vs. cm).

Then run `run_gate` once with the real baseline model + a clean batch (expect an overall **PASS**),
and once with your bad model + your corrupted batch (expect an overall **FAIL**, and make sure the
printed summary tells you *which* checks failed and why).

In [ ]:
# TODO: build bad_recall_model by training on an artificially malignant-depleted subset of X_train/y_train

# TODO: build corrupted_batch by copying X_test and scaling one column by a large factor

# TODO: run_gate(baseline_model, MODEL_PATH, X_test, y_test, BASELINE_METRICS, X_test)                 # expect PASS
# TODO: run_gate(bad_recall_model, MODEL_PATH, X_test, y_test, BASELINE_METRICS, corrupted_batch)       # expect FAIL


---

## Solution

*(scroll down when you're ready — try it yourself first)*

<details>
<summary>Click to reveal solution</summary>

**Design notes before the code:**

- `validate_schema` checks columns, dtype, nulls, and range *in that order* so the violation list
  reads like a triage log — a missing column is reported before a range nitpick on a column that's
  actually there.
- `check_regression` only gates on **recall**, deliberately ignoring whether accuracy improved —
  that's the whole point of the exercise: a model can look better in aggregate while quietly
  getting worse at the one thing that matters most.
- `check_determinism` reloads from disk rather than comparing two in-memory copies, because the
  failure mode it's guarding against is specifically about serialization (pickling a fitted
  `StandardScaler`/estimator and getting it back bit-for-bit, not about training randomness —
  `RANDOM_STATE` already pins that separately).
- The bad candidate is built by depleting malignant examples from the *training* set only — the
  test set stays untouched, so the recall drop we measure is a real generalization effect, not an
  artifact of evaluating on a different distribution.

```python
def validate_schema(df_in, schema=SCHEMA):
    violations = []
    expected_cols = list(schema.keys())
    missing = [c for c in expected_cols if c not in df_in.columns]
    extra = [c for c in df_in.columns if c not in expected_cols]
    if missing:
        violations.append(f"missing columns: {missing}")
    if extra:
        violations.append(f"unexpected columns: {extra}")
    for col in expected_cols:
        if col not in df_in.columns:
            continue
        series = df_in[col]
        if not pd.api.types.is_float_dtype(series) and not pd.api.types.is_integer_dtype(series):
            violations.append(f"column '{col}' has dtype {series.dtype}, expected numeric")
            continue
        if series.isnull().any():
            violations.append(f"column '{col}' has {series.isnull().sum()} null value(s)")
        lo, hi = schema[col]["min"], schema[col]["max"]
        out_of_range = series[(series < lo) | (series > hi)]
        if len(out_of_range) > 0:
            violations.append(
                f"column '{col}' has {len(out_of_range)} value(s) outside training range "
                f"[{lo:.3f}, {hi:.3f}] (e.g. {out_of_range.iloc[0]:.3f})"
            )
    return {"passed": len(violations) == 0, "violations": violations}


def check_regression(candidate_model, baseline_metrics, X_test, y_test, min_recall_drop=-0.02):
    preds = candidate_model.predict(X_test)
    candidate_metrics = {
        "recall": recall_score(y_test, preds),
        "precision": precision_score(y_test, preds),
        "accuracy": accuracy_score(y_test, preds),
    }
    recall_delta = candidate_metrics["recall"] - baseline_metrics["recall"]
    passed = recall_delta >= min_recall_drop
    reason = (
        f"recall {candidate_metrics['recall']:.4f} vs baseline {baseline_metrics['recall']:.4f} "
        f"(delta {recall_delta:+.4f}, allowed drop {min_recall_drop:+.4f})"
    )
    return {
        "passed": passed,
        "candidate_metrics": candidate_metrics,
        "baseline_metrics": baseline_metrics,
        "reason": reason,
    }


def check_determinism(model_path, X_sample):
    reloaded = joblib.load(model_path)
    in_memory_preds = baseline_model.predict(X_sample)
    reloaded_preds = reloaded.predict(X_sample)
    mismatches = int(np.sum(in_memory_preds != reloaded_preds))
    return {"passed": mismatches == 0, "mismatches": mismatches}


def run_gate(candidate_model, model_path, X_test, y_test, baseline_metrics, inference_batch, schema=SCHEMA):
    schema_result = validate_schema(inference_batch, schema)
    regression_result = check_regression(candidate_model, baseline_metrics, X_test, y_test)
    determinism_result = check_determinism(model_path, X_test.iloc[:20])
    overall_passed = schema_result["passed"] and regression_result["passed"] and determinism_result["passed"]

    print("=== Model Promotion Gate ===")
    print(f"Schema check:      {'PASS' if schema_result['passed'] else 'FAIL'}")
    print(f"Regression check:  {'PASS' if regression_result['passed'] else 'FAIL'} ({regression_result['reason']})")
    print(f"Determinism check: {'PASS' if determinism_result['passed'] else 'FAIL'}")
    print(f"OVERALL:           {'PASS' if overall_passed else 'FAIL'}")

    return {
        "overall_passed": overall_passed,
        "schema": schema_result,
        "regression": regression_result,
        "determinism": determinism_result,
    }


# --- adversarial fixtures ---

# 1. A candidate trained on a malignant-depleted subsample of the training data only.
bad_malignant_idx = y_train[y_train == 1].sample(frac=0.15, random_state=RANDOM_STATE).index
skewed_idx = X_train.index.difference(y_train[y_train == 1].index).union(bad_malignant_idx)
X_train_skewed, y_train_skewed = X_train.loc[skewed_idx], y_train.loc[skewed_idx]

bad_recall_model = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])
bad_recall_model.fit(X_train_skewed, y_train_skewed)

# 2. A corrupted inference batch: "mean radius" scaled 10x (e.g. a unit mixup upstream).
corrupted_batch = X_test.copy()
corrupted_batch["mean radius"] = corrupted_batch["mean radius"] * 10

print("Expect PASS:")
run_gate(baseline_model, MODEL_PATH, X_test, y_test, BASELINE_METRICS, X_test)

print("\nExpect FAIL:")
run_gate(bad_recall_model, MODEL_PATH, X_test, y_test, BASELINE_METRICS, corrupted_batch)
```

Running this prints an overall **PASS** for the real baseline model against a clean batch, and an
overall **FAIL** for the bad candidate against the corrupted batch — with the printed summary
showing the regression check failing on recall (0.92 → 0.81, well past the 2-point tolerance) and
the schema check failing specifically on `mean radius` being far outside its contracted range.
Accuracy for the bad candidate can still look fine in isolation (precision even improves, since a
model that calls fewer cases malignant makes fewer false-positive mistakes) — which is exactly the
trap a recall-blind regression test would miss.

</details>